# Train DataLoader 검증 노트북 (Colab 완결판)

`new_dataset.py` / `new_datamodule.py` 검증용. **셀 순서대로** 실행하세요.

| Step | 내용 | 필수 |
|------|------|------|
| **Colab** | Drive 마운트 + `git clone`/`git pull` + pip | Colab만 |
| 0 | 환경 설정 (`data/` → Drive) | ✓ |
| helpers | 다운로드/압축 유틸 | ✓ |
| 1 | Zenodo dev set → `data/dev_set/` | dataloader 스모크 시 |
| 2 | SpAudSyn | dataloader 스모크 시 |
| 3 | Semantic Hearing 다운로드 확인 (tar 압축 해제) | 선택 |
| 3b | EARS 다운로드 | 선택 |
| 4 | `add_data.sh` 실행 → dev_set 보강 | 선택 |
| 4-debug | SEMHEAR_DIR 내부 구조 확인 | 문제 발생 시 |
| 5 | 경로 검증 | dataloader 스모크 시 |
| **unittest** | **`test_new_*.py` 전용 셀** | ✓ (데이터 불필요) |
| 7–8 | train/val dataloader 스모크 | Step 1–2·5 후 |

> **baseline 데이터 준비 순서**: Colab → 0 → helpers → 1 → 2 → 3 → 3b → 4 → 5 → unittest → 7 → 8
>
> **baseline 공식**: `bash add_data.sh --semhear_path /path/to/BinauralCuratedDataset --ears_path /path/to/EARS`
>
> **unittest만**: Colab → 0 → helpers → **unittest** 셀 (Step 1–5 skip 가능)

## [Colab] Drive 마운트 + repo clone / pull

로컬 PC면 **skip**. Repo 최신 반영: `git pull`

In [4]:
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/Mockdd/dcase2026_task4.git"
COLAB_REPO_DIR = Path("/content/dcase2026_task4")
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/dcase2026_task4/data")

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)

    if (COLAB_REPO_DIR / ".git").exists():
        print("[git pull]")
        subprocess.run(["git", "pull"], cwd=COLAB_REPO_DIR, check=True)
    elif not (COLAB_REPO_DIR / "src" / "datamodules" / "new_dataset.py").exists():
        print("[git clone]", REPO_URL)
        subprocess.run(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)], check=True)

    %cd {COLAB_REPO_DIR}
    !pip install -q lightning pyyaml soundfile librosa scipy python-sofa tqdm
    !git log -1 --oneline
    print("\nColab OK | code:", COLAB_REPO_DIR)
    print("         | data:", DRIVE_DATA_DIR)
else:
    print("Not Colab — skip this cell.")

Mounted at /content/drive
[git clone] https://github.com/Mockdd/dcase2026_task4.git
/content/dcase2026_task4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 106.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.7/270.7 kB 26.1 MB/s eta 0:00:00
49d76b5 (HEAD -> main, origin/main, origin/HEAD) unittest wrong expectation

Colab OK | code: 

## Step 0. 환경 설정

In [5]:
import os
import re
import shutil
import subprocess
import sys
import tarfile
import unittest
import urllib.request
import zipfile
from io import StringIO
from pathlib import Path

import torch
import yaml

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def _data_is_scaffold_only(data_path: Path) -> bool:
    if not data_path.exists():
        return True
    return not any(p.is_file() and p.stat().st_size > 0 for p in data_path.rglob("*"))


if IN_COLAB:
    PROJECT_ROOT = Path("/content/dcase2026_task4")
    DATA_DIR = Path("/content/drive/MyDrive/dcase2026_task4/data")
else:
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / "src" / "datamodules" / "new_dataset.py").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    DATA_DIR = PROJECT_ROOT / "data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

LOCAL_DATA = PROJECT_ROOT / "data"
if IN_COLAB:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if LOCAL_DATA.is_symlink() and LOCAL_DATA.resolve() == DATA_DIR.resolve():
        print("[ok] symlink already set")
    elif _data_is_scaffold_only(LOCAL_DATA):
        if LOCAL_DATA.exists() or LOCAL_DATA.is_symlink():
            LOCAL_DATA.unlink() if LOCAL_DATA.is_symlink() else shutil.rmtree(LOCAL_DATA)
        LOCAL_DATA.symlink_to(DATA_DIR, target_is_directory=True)
        print("[ok] symlink created")
    else:
        shutil.copytree(LOCAL_DATA, DATA_DIR, dirs_exist_ok=True)
        shutil.rmtree(LOCAL_DATA)
        LOCAL_DATA.symlink_to(DATA_DIR, target_is_directory=True)
        print("[ok] local data moved to Drive + symlink")
    print("symlink:", LOCAL_DATA, "->", DATA_DIR.resolve())

DEV_SET_DIR = DATA_DIR / "dev_set"
DOWNLOAD_DIR = DATA_DIR / "downloads"
ZENODO_DIR = DOWNLOAD_DIR / "zenodo"
SEMHEAR_TAR = DATA_DIR / "BinauralCuratedDataset.tar"
SEMHEAR_DIR = DATA_DIR / "BinauralCuratedDataset"
SPAUDSYN_DIR = PROJECT_ROOT / "src" / "modules" / "spatial_audio_synthesizer"

for d in (DATA_DIR, DOWNLOAD_DIR, ZENODO_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("DATA_DIR    :", DATA_DIR.resolve())
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

[ok] local data moved to Drive + symlink
symlink: /content/dcase2026_task4/data -> /content/drive/MyDrive/dcase2026_task4/data
PROJECT_ROOT: /content/dcase2026_task4
DATA_DIR    : /content/drive/MyDrive/dcase2026_task4/data
torch: 2.11.0+cu128 | cuda: True


## helpers. 다운로드 / Zenodo split-zip 유틸

In [6]:
def describe_batch(batch, title="batch"):
    print(f"\n=== {title} ===")
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            print(f"  {k:20s} shape={tuple(v.shape)} dtype={v.dtype}")
        elif isinstance(v, list):
            print(f"  {k:20s} list[len={len(v)}]")
        else:
            print(f"  {k:20s} {type(v).__name__}")


def download_url(url: str, dest: Path, label: str = "") -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[skip] {label or dest.name}")
        return
    print(f"[download] {label or dest.name}")
    def _progress(n, size, total):
        if total > 0:
            print(f"\r  {min(n*size,total)/total*100:5.1f}%", end="", flush=True)
    urllib.request.urlretrieve(url, dest, reporthook=_progress)
    print("\n  done.")


def _ensure_archive_tools():
    pkgs = [("unzip", "unzip"), ("zip", "zip"), ("p7zip-full", "7z")]
    need = [pkg for pkg, cmd in pkgs if not shutil.which(cmd)]
    if need:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", *need], check=True)


def _unzip_test(path: Path) -> bool:
    return subprocess.run(["unzip", "-t", str(path)], capture_output=True).returncode == 0


def _extract_with_unzip(zip_path: Path, dest_dir: Path) -> None:
    dest_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["unzip", "-o", "-q", str(zip_path), "-d", str(dest_dir)], check=True)


def _extract_with_7z(zip_path: Path, dest_dir: Path) -> None:
    dest_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["7z", "x", "-y", f"-o{dest_dir}", zip_path.name], cwd=zip_path.parent, check=True)


def prepare_zenodo_dev_set(zenodo_dir: Path, extract_root: Path) -> None:
    """Zenodo split zip → dev_set. zip -s 0 / unzip / 7z fallback. Python zipfile 사용 안 함."""
    if (DEV_SET_DIR / "metadata" / "valid.json").exists():
        print("[skip] dev_set already installed")
        return

    _ensure_archive_tools()
    split_zip = zenodo_dir / "DCASE2026Task4Dataset.zip"
    merged_zip = zenodo_dir / "DCASE2026Task4DatasetFull.zip"
    if not split_zip.exists():
        raise FileNotFoundError(f"Missing {split_zip}")

    if merged_zip.exists() and not _unzip_test(merged_zip):
        print(f"[cleanup] bad merged zip: {merged_zip.name}")
        merged_zip.unlink()

    marker = extract_root / ".extracted"
    if marker.exists() and not any(extract_root.rglob("**/valid.json")):
        marker.unlink()
    if extract_root.exists() and not any(extract_root.rglob("**/valid.json")):
        shutil.rmtree(extract_root)

    if _extract_done(extract_root):
        print("[skip] extract folder ready")
    else:
        extract_root.mkdir(parents=True, exist_ok=True)
        ok = False
        print("[extract A] unzip split archive")
        try:
            _extract_with_unzip(split_zip, extract_root)
            ok = _extract_done(extract_root)
        except subprocess.CalledProcessError as e:
            print("  failed:", e)
        if not ok:
            if not merged_zip.exists():
                print("[merge] zip -s 0")
                subprocess.run(
                    ["zip", "-s", "0", split_zip.name, "--out", merged_zip.name],
                    cwd=zenodo_dir, check=True,
                )
            print("[extract B] unzip merged")
            _extract_with_unzip(merged_zip, extract_root)
            ok = _extract_done(extract_root)
        if not ok:
            print("[extract C] 7z")
            if extract_root.exists():
                shutil.rmtree(extract_root)
            extract_root.mkdir(parents=True)
            _extract_with_7z(split_zip, extract_root)
            ok = _extract_done(extract_root)
        if not ok:
            raise RuntimeError("Zenodo extract failed — check .z01-.z10 downloads")
        marker.write_text("ok", encoding="utf-8")

    install_dev_set_from_extract(extract_root)


def _extract_done(extract_root: Path) -> bool:
    return any(extract_root.rglob("**/valid.json")) if extract_root.exists() else False


def install_dev_set_from_extract(extract_root: Path) -> None:
    if (DEV_SET_DIR / "metadata" / "valid.json").exists():
        return
    candidates = [
        extract_root / "DCASE2026Task4Dataset" / "data" / "dev_set",
        extract_root / "data" / "dev_set",
    ]
    candidates += [p for p in extract_root.rglob("dev_set") if (p / "metadata" / "valid.json").exists()]
    src = next((p for p in candidates if p.exists()), None)
    if src is None:
        raise FileNotFoundError(f"dev_set not found under {extract_root}")
    print(f"[install] {src} -> {DEV_SET_DIR}")
    if DEV_SET_DIR.exists():
        shutil.rmtree(DEV_SET_DIR)
    shutil.copytree(src, DEV_SET_DIR)

print("helpers loaded.")

helpers loaded.


## Step 1. Zenodo DCASE dev set

[Zenodo 19328046](https://zenodo.org/records/19328046) — Drive `data/downloads/zenodo/`에 저장

In [19]:
ZENODO_URLS = [
    line.strip()
    for line in (PROJECT_ROOT / "dev_set_zenodo.txt").read_text(encoding="utf-8").splitlines()
    if line.strip() and "lisence.pdf" not in line.strip().lower()
]
for url in ZENODO_URLS:
    fname = url.rsplit("/", 1)[-1]
    download_url(url, ZENODO_DIR / fname, label=fname)

extract_root = ZENODO_DIR / "extracted"
prepare_zenodo_dev_set(ZENODO_DIR, extract_root)
assert (DEV_SET_DIR / "metadata" / "valid.json").exists()
print("\nStep 1 OK:", DEV_SET_DIR)

[skip] DCASE2026Task4Dataset.z01
[skip] DCASE2026Task4Dataset.z02
[skip] DCASE2026Task4Dataset.z03
[skip] DCASE2026Task4Dataset.z04
[skip] DCASE2026Task4Dataset.z05
[skip] DCASE2026Task4Dataset.z06
[skip] DCASE2026Task4Dataset.z07
[skip] DCASE2026Task4Dataset.z08
[skip] DCASE2026Task4Dataset.z09
[skip] DCASE2026Task4Dataset.z10
[skip] DCASE2026Task4Dataset.zip


KeyboardInterrupt: 

## Step 2. SpAudSyn

In [7]:
marker = SPAUDSYN_DIR / "spatial_audio_synthesizer.py"
if marker.exists():
    print(f"[skip] {SPAUDSYN_DIR}")
else:
    clone_dir = DOWNLOAD_DIR / "SpAudSyn_repo"
    if clone_dir.exists():
        shutil.rmtree(clone_dir)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/nttcslab/SpAudSyn.git", str(clone_dir)], check=True)
    SPAUDSYN_DIR.parent.mkdir(parents=True, exist_ok=True)
    if SPAUDSYN_DIR.exists():
        shutil.rmtree(SPAUDSYN_DIR)
    shutil.copytree(clone_dir / "src", SPAUDSYN_DIR)
    print(f"[install] {SPAUDSYN_DIR}")
assert marker.exists()
print("\nStep 2 OK")

[install] /content/dcase2026_task4/src/modules/spatial_audio_synthesizer

Step 2 OK


## Step 3. Semantic Hearing 압축 해제 확인

[SemanticHearing](https://github.com/vb000/SemanticHearing) — `BinauralCuratedDataset.tar`를 Drive에 미리 다운로드해두세요.

```
wget -P data https://semantichearing.cs.washington.edu/BinauralCuratedDataset.tar
```

이 셀은 tar가 이미 Drive에 있으면 압축 해제하고, 없으면 다운로드 후 해제합니다.

In [8]:
# Step 3: Semantic Hearing tar 압축 해제
import tarfile

SEMHEAR_URL = "https://semantichearing.cs.washington.edu/BinauralCuratedDataset.tar"

if SEMHEAR_DIR.exists() and any(SEMHEAR_DIR.iterdir()):
    print(f"[skip] 이미 압축 해제됨: {SEMHEAR_DIR}")
elif SEMHEAR_TAR.exists() and SEMHEAR_TAR.stat().st_size > 0:
    print(f"[extract] {SEMHEAR_TAR} → {DATA_DIR}")
    with tarfile.open(SEMHEAR_TAR, "r") as tar:
        tar.extractall(DATA_DIR)
    print("[extract] 완료")
else:
    print(f"[download+extract] {SEMHEAR_URL}")
    download_url(SEMHEAR_URL, SEMHEAR_TAR, label="BinauralCuratedDataset.tar")
    with tarfile.open(SEMHEAR_TAR, "r") as tar:
        tar.extractall(DATA_DIR)
    print("[extract] 완료")

# BinauralCuratedDataset이 아닌 이름으로 풀렸을 경우 탐지
if not SEMHEAR_DIR.exists():
    alt = next((p for p in DATA_DIR.glob("Binaural*") if p.is_dir()), None)
    if alt:
        SEMHEAR_DIR = alt
        print(f"[info] SEMHEAR_DIR → {SEMHEAR_DIR}")

assert SEMHEAR_DIR.exists(), f"SEMHEAR_DIR not found: {SEMHEAR_DIR}"

# 내부 구조 출력 (add_data.sh 디버깅용)
print(f"\nSEMHEAR_DIR: {SEMHEAR_DIR}")
print("내부 항목:")
for p in sorted(SEMHEAR_DIR.iterdir()):
    marker = "[dir]" if p.is_dir() else "[file]"
    print(f"  {marker} {p.name}")

print("\nStep 3 OK:", SEMHEAR_DIR)

[skip] 이미 압축 해제됨: /content/drive/MyDrive/dcase2026_task4/data/BinauralCuratedDataset

SEMHEAR_DIR: /content/drive/MyDrive/dcase2026_task4/data/BinauralCuratedDataset
내부 항목:
  [dir] FSD50K
  [dir] TAU-acoustic-sounds
  [dir] bg_scaper_fmt

Step 3 OK: /content/drive/MyDrive/dcase2026_task4/data/BinauralCuratedDataset


## Step 3b. EARS 다운로드 (선택)

[EARS dataset](https://github.com/facebookresearch/ears_dataset) — speech 보강용 (p001 ~ p107, 각 ~수십MB).
전체 다운로드는 시간이 오래 걸립니다. `MAX_IDX`를 줄이면 일부만 받습니다.

In [9]:
# Step 3b: EARS 다운로드
# 전체: MAX_IDX = 107 / 빠른 테스트: MAX_IDX = 3
MAX_IDX = 3

EARS_DIR = DATA_DIR / "EARS"
EARS_MARKER = EARS_DIR / ".ears_downloaded"

if EARS_MARKER.exists():
    print(f"[skip] EARS already downloaded: {EARS_DIR}")
else:
    EARS_DIR.mkdir(parents=True, exist_ok=True)
    BASE = "https://github.com/facebookresearch/ears_dataset/releases/download/dataset"
    for idx in range(1, MAX_IDX + 1):
        name = f"p{idx:03d}"
        zip_path = EARS_DIR / f"{name}.zip"
        person_dir = EARS_DIR / name
        if person_dir.exists() and any(person_dir.iterdir()):
            print(f"[skip] {name}")
            continue
        download_url(f"{BASE}/{name}.zip", zip_path, label=f"{name}.zip")
        import zipfile
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(EARS_DIR)
        zip_path.unlink()
    EARS_MARKER.write_text("ok", encoding="utf-8")
    print(f"EARS 다운로드 완료: {EARS_DIR}")

print("Step 3b OK:", EARS_DIR)

[skip] EARS already downloaded: /content/drive/MyDrive/dcase2026_task4/data/EARS
Step 3b OK: /content/drive/MyDrive/dcase2026_task4/data/EARS


## Step 4. add_data.sh 실행 — dev_set 보강

baseline의 `add_data.sh`를 Python에서 직접 호출합니다.
내부적으로 두 스크립트를 순서대로 실행합니다:

| 스크립트 | 입력 | 출력 | 내용 |
|----------|------|------|------|
| `add_interference.py` | `SEMHEAR_DIR` (bg set) | `dev_set/interference/` | Semantic Hearing non-target 클래스 → interference |
| `add_sound_event.py` | `SEMHEAR_DIR/FSD50K` + `EARS_DIR` | `dev_set/sound_event/` | FSD50K + EARS → target sound event 보강 |

> **참고**: baseline README 명령어
> ```bash
> bash add_data.sh --semhear_path /path/to/BinauralCuratedDataset --ears_path /path/to/EARS
> ```

In [10]:
import zipfile

BASE = "https://zenodo.org/record/4060432/files"
FSD_DIR = SEMHEAR_DIR / "FSD50K"

for fname in ["FSD50K.metadata.zip", "FSD50K.ground_truth.zip"]:
    dest = FSD_DIR / fname
    if not (FSD_DIR / fname.replace(".zip", "")).exists():
        download_url(f"{BASE}/{fname}?download=1", dest, label=fname)
        with zipfile.ZipFile(dest) as zf:
            zf.extractall(FSD_DIR)
        dest.unlink()
        print(f"[ok] {fname}")
    else:
        print(f"[skip] {fname}")

# 구조 확인
for p in sorted(FSD_DIR.iterdir()):
    print(p.name)

[skip] FSD50K.metadata.zip
[skip] FSD50K.ground_truth.zip
FSD50K.dev_audio
FSD50K.ground_truth
FSD50K.metadata


In [17]:
os.listdir('/content/drive/MyDrive/dcase2026_task4/data/BinauralCuratedDataset/FSD50K/FSD50K.metadata')

['eval_clips_info_FSD50K.json',
 'pp_pnp_ratings_FSD50K.json',
 'class_info_FSD50K.json',
 'dev_clips_info_FSD50K.json',
 'collection']

In [12]:
import json

meta_dir = FSD_DIR / "FSD50K.metadata"

# class_info 구조 확인
with open(meta_dir / "class_info_FSD50K.json") as f:
    class_info = json.load(f)
# 첫 3개 항목
for k, v in list(class_info.items())[:3]:
    print(k, "→", v)

print("---")

# dev_clips_info 구조 확인
with open(meta_dir / "dev_clips_info_FSD50K.json") as f:
    clips_info = json.load(f)
# 첫 3개 항목
for k, v in list(clips_info.items())[:3]:
    print(k, "→", v)

/m/0dv3j → {'faq': '<div class="ui accordion">\r\n\r\n class="title">\r\n  \t<i class="dropdown icon"></i>\r\n  \tAre frying, hissing or sizzling sounds considered ‘Boiling’ sounds?\r\n\t</div>\r\n\t class="content">\r\n  \t<p>No.</p>\r\n\t</div>\r\n\r\n class="title">\r\n  \t<i class="dropdown icon"></i>\r\n  \tAre ‘Boiling’ sounds containing hissing sounds from gas stoves or kettles considered <i>Present and predominant</i> sounds?\r\n\t</div>\r\n\t class="content">\r\n  \t<p>No. When ‘Boiling’ and hissing are both salient sounds, please choose <i>Present but not predominant</i>.</p> \r\n\t</div>\r\n\r\n class="title">\r\n  \t<i class="dropdown icon"></i>\r\n  \tAre kettle whistle and pressure cooker whistle sounds considered ‘Boiling’ sounds?\r\n\t</div>\r\n\t class="content">\r\n  \t<p>No.</p>\r\n\t</div>\r\n\r\n class="title">\r\n  \t<i class="dropdown icon"></i>\r\n  \tAre bubbling sounds created by blowing water with the mouth or a straw considered ‘Boiling’ sounds?\r\n\t</div>\

In [15]:
gt_dir = FSD_DIR / "FSD50K.ground_truth"
for p in sorted(gt_dir.iterdir()):
    print(p.name)

# 파일 내용 확인
for p in gt_dir.iterdir():
    print(f"\n=== {p.name} ===")
    if p.suffix == ".json":
        with open(p) as f:
            d = json.load(f)
        for k, v in list(d.items())[:2]:
            print(k, "→", v)
    elif p.suffix == ".csv":
        import pandas as pd
        df = pd.read_csv(p, nrows=3)
        print(df.columns.tolist())
        print(df.head(3))

dev.csv
eval.csv
vocabulary.csv

=== vocabulary.csv ===
['0', 'Accelerating_and_revving_and_vroom', '/m/07q2z82']
   0 Accelerating_and_revving_and_vroom  /m/07q2z82
0  1                          Accordion     /m/0mkg
1  2                    Acoustic_guitar  /m/042v_gx
2  3                           Aircraft     /m/0k5j

=== dev.csv ===
['fname', 'labels', 'mids', 'split']
   fname                                             labels  \
0  64760  Electric_guitar,Guitar,Plucked_string_instrume...   
1  16399  Electric_guitar,Guitar,Plucked_string_instrume...   
2  16401  Electric_guitar,Guitar,Plucked_string_instrume...   

                                            mids  split  
0  /m/02sgy,/m/0342h,/m/0fx80y,/m/04szw,/m/04rlf  train  
1  /m/02sgy,/m/0342h,/m/0fx80y,/m/04szw,/m/04rlf  train  
2  /m/02sgy,/m/0342h,/m/0fx80y,/m/04szw,/m/04rlf  train  

=== eval.csv ===
['fname', 'labels', 'mids']
    fname                                             labels  \
0   37199  Electric_guitar,Gu

In [30]:
import os
import pandas as pd

# bg_scaper_fmt를 로컬에 생성 (Drive X, 로컬 /content에)
bg_dir = Path("/content/bg_scaper_fmt")

# Cell 19에서 df가 eval.csv로 덮어써질 수 있으므로 dev.csv로 명시 로드
gt_dir = SEMHEAR_DIR / "FSD50K" / "FSD50K.ground_truth"
df = pd.read_csv(gt_dir / "dev.csv")

copied = 0
skipped_no_wav = 0
skipped_no_class = 0

# add_interference.py와 동일한 selected_classes 목록
selected_classes = [
    "Air conditioning", "Aircraft", "Bird flight, flapping wings", "Bleat", "Boiling", "Boom",
    "Burping, eructation", "Burst, pop", "Bus", "Camera", "Car passing by",
    "Cattle, bovinae", "Chainsaw", "Chewing, mastication", "Chink, clink", "Clip-clop",
    "Cluck", "Clunk", "Coin (dropping)", "Crack", "Crackle", "Creak", "Croak", "Crow",
    "Crumpling, crinkling", "Crushing", "Drill", "Drip", "Electric toothbrush", "Engine",
    "Fart", "Finger snapping", "Fire", "Fire alarm", "Firecracker", "Fireworks",
    "Fixed-wing aircraft, airplane", "Frog", "Gears", "Growling", "Gurgling", "Helicopter",
    "Hiss", "Hoot", "Howl", "Howl (wind)", "Jackhammer", "Keys jangling", "Lawn mower",
    "Light engine (high frequency)", "Microwave oven", "Moo", "Oink",
    "Packing tape, duct tape", "Pig", "Printer", "Purr", "Rain", "Rain on surface",
    "Raindrop", "Ratchet, pawl", "Rattle", "Sanding", "Sawing", "Scissors", "Screech",
    "Sheep", "Ship", "Shuffling cards", "Skateboard", "Slam", "Sliding door", "Sneeze",
    "Sniff", "Snoring", "Splinter", "Squeak", "Stream", "Subway, metro, underground",
    "Tap", "Tearing", "Thump, thud", "Tick", "Tick-tock", "Toothbrush",
    "Traffic noise, roadway noise", "Train", "Train horn",
    "Velcro, hook and loop fastener", "Waterfall", "Whoosh, swoosh, swish", "Wind",
    "Writing", "Zipper (clothing)",
]
dev_audio_dir = SEMHEAR_DIR / "FSD50K" / "FSD50K.dev_audio"
split_map = {"train": "train", "val": "valid"}
normalize = lambda s: s.replace("_", " ").strip()
selected_set = set(selected_classes)

In [31]:
if (DEV_SET_DIR / "interference" / "train").exists():
    print("[skip] interference/train already exists — FSD50K copy skip")
else:
    for _, row in df.iterrows():
        fname = str(row["fname"])
        split = split_map.get(row["split"])
        if split is None:
            continue

        wav_src = dev_audio_dir / f"{fname}.wav"
        if not wav_src.exists():
            skipped_no_wav += 1
            continue

        labels = [normalize(l.strip()) for l in str(row["labels"]).split(",")]
        matched = [l for l in labels if l in selected_set]
        if not matched:
            skipped_no_class += 1
            continue

        for label in matched:
            dest_dir = bg_dir / split / label
            dest_dir.mkdir(parents=True, exist_ok=True)
            dest = dest_dir / f"{fname}.wav"
            if not dest.exists():
                shutil.copy2(wav_src, dest)  # Drive는 symlink 미지원
                copied += 1

print(f"심링크 생성: {copied}")
print(f"wav 없음 skip: {skipped_no_wav}")
print(f"selected_class 미해당 skip: {skipped_no_class}")

# 커버된 클래스 확인
covered = set()
for split in ("train", "valid"):
    split_dir = bg_dir / split
    if split_dir.exists():
        covered |= {p.name for p in split_dir.iterdir() if p.is_dir()}

missing = selected_set - covered
print(f"\n커버된 클래스: {len(covered)}/94")
if missing:
    print(f"FSD50K에 없는 클래스 ({len(missing)}개):")
    for c in sorted(missing):
        print(f"  - {c}")

NameError: name 'split_map' is not defined

In [21]:
result = subprocess.run([
    sys.executable, "add_interference.py",
    "--input_dir", str(bg_dir),
    "--output_dir", str(DEV_SET_DIR / "interference"),
    "--target_sr=32000", "--amp_threshold=0.02",
    "--min_length=0.1", "--segment=10", "--shift=0.1",
], cwd=PROJECT_ROOT, capture_output=True, text=True)

print(result.stdout[-3000:] if result.stdout else "(no stdout)")
if result.returncode != 0:
    print("[STDERR]", result.stderr[-2000:])
else:
    print("add_interference.py OK")

input_dir    : /content/bg_scaper_fmt
output_dir   : /content/drive/MyDrive/dcase2026_task4/data/dev_set/interference
output_sr    : 32000
amp_threshold: 0.02
min_length   : 0.1
segment      : 10.0
shift        : 0.1
input_dir    : /content/bg_scaper_fmt/train
output_dir   : /content/drive/MyDrive/dcase2026_task4/data/dev_set/interference/train

[STDERR] Traceback (most recent call last):
  File "/content/dcase2026_task4/add_interference.py", line 147, in <module>
    prepare_data(input_dir= os.path.join(args.input_dir, 'train'),
  File "/content/dcase2026_task4/add_interference.py", line 96, in prepare_data
    assert os.path.exists(src_path) and os.path.isdir(src_path), f'There is no {src_path}'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: There is no /content/bg_scaper_fmt/train/Air conditioning



In [22]:
result = subprocess.run([
    sys.executable, "add_interference.py",
    "--input_dir", str(bg_dir),
    "--output_dir", str(DEV_SET_DIR / "interference"),
    "--target_sr=32000", "--amp_threshold=0.02",
    "--min_length=0.1", "--segment=10", "--shift=0.1",
], cwd=PROJECT_ROOT, capture_output=True, text=True)

print(result.stdout[-3000:] if result.stdout else "(no stdout)")
if result.returncode != 0:
    print("[STDERR]", result.stderr[-2000:])
else:
    print("add_interference.py OK")

input_dir    : /content/bg_scaper_fmt
output_dir   : /content/drive/MyDrive/dcase2026_task4/data/dev_set/interference
output_sr    : 32000
amp_threshold: 0.02
min_length   : 0.1
segment      : 10.0
shift        : 0.1
input_dir    : /content/bg_scaper_fmt/train
output_dir   : /content/drive/MyDrive/dcase2026_task4/data/dev_set/interference/train

[STDERR] Traceback (most recent call last):
  File "/content/dcase2026_task4/add_interference.py", line 147, in <module>
    prepare_data(input_dir= os.path.join(args.input_dir, 'train'),
  File "/content/dcase2026_task4/add_interference.py", line 96, in prepare_data
    assert os.path.exists(src_path) and os.path.isdir(src_path), f'There is no {src_path}'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: There is no /content/bg_scaper_fmt/train/Air conditioning



In [15]:
!cat /content/dcase2026_task4/add_data.sh

#!/bin/bash

while [[ "$#" -gt 0 ]]; do
    case "$1" in
        --semhear_path)
            SEMHEAR_PATH="$2"
            shift 2
            ;;
        --ears_path)
            EARS_PATH="$2"
            shift 2
            ;;
        *)
            echo "Unknown parameter passed: $1"
            exit 1
            ;;
    esac
done

TARGET_SR=32000
AMP_THRESHOLD=0.02
MIN_LENGTH=0.1
SEGMENT=10
SHIFT=0.1

python add_interference.py \
    --input_dir="${SEMHEAR_PATH}/bg_scaper_fmt" \
    --output_dir=data/dev_set/interference \
    --target_sr=$TARGET_SR \
    --amp_threshold=$AMP_THRESHOLD \
    --min_length=$MIN_LENGTH \
    --segment=$SEGMENT \
    --shift=$SHIFT

python add_sound_event.py \
    --input_dir="${SEMHEAR_PATH}/FSD50K" \
    --output_dir=data/dev_set/sound_event \
    --pickup_json=data/dev_set/config/FSD50K_config.json \
    --target_sr=$TARGET_SR \
    --amp_threshold=$AMP_THRESHOLD \
    --min_length=$MIN_LENGTH \
    --segment=$SEGMENT \
    --shift=$SHIFT

python ad

In [26]:
!head -60 /content/dcase2026_task4/add_interference.py

import os
import shutil
from tqdm import tqdm
import soundfile as sf
import argparse
import librosa
import numpy as np
import json

selected_classes = ["Air conditioning", "Aircraft", "Bird flight, flapping wings", "Bleat", "Boiling", "Boom",
              "Burping, eructation", "Burst, pop", "Bus", "Camera", "Car passing by",
              "Cattle, bovinae", "Chainsaw", "Chewing, mastication", "Chink, clink", "Clip-clop",
              "Cluck", "Clunk", "Coin (dropping)", "Crack", "Crackle", "Creak", "Croak", "Crow",
              "Crumpling, crinkling", "Crushing", "Drill", "Drip", "Electric toothbrush", "Engine",
              "Fart", "Finger snapping", "Fire", "Fire alarm", "Firecracker", "Fireworks", "Fixed-wing aircraft, airplane",
              "Frog", "Gears", "Growling", "Gurgling", "Helicopter", "Hiss", "Hoot", "Howl", "Howl (wind)",
              "Jackhammer", "Keys jangling", "Lawn mower", "Light engine (high frequency)", "Microwave oven",
              "Moo", "Oink", "Pack

In [23]:
import json

# FSD50K 메타데이터로 어떤 클래스가 있는지 확인
fsd_dir = SEMHEAR_DIR / "FSD50K"
# vocab 파일 또는 클래스 폴더 목록 확인
print("FSD50K 내부 구조:")
for p in sorted(fsd_dir.iterdir()):
    print(f"  {p.name}")

FSD50K 내부 구조:
  FSD50K.dev_audio
  FSD50K.ground_truth
  FSD50K.metadata


In [24]:
fsd_dir = SEMHEAR_DIR / "FSD50K"
for p in sorted(fsd_dir.iterdir()):
    print(p.name)
    if p.is_dir():
        # 하위 파일 5개만
        for pp in list(p.iterdir())[:5]:
            print(f"  {pp.name}")

FSD50K.dev_audio
  70361.wav
  356547.wav
  251886.wav
  200336.wav
  50369.wav
FSD50K.ground_truth
  vocabulary.csv
  dev.csv
  eval.csv
FSD50K.metadata
  eval_clips_info_FSD50K.json
  pp_pnp_ratings_FSD50K.json
  class_info_FSD50K.json
  dev_clips_info_FSD50K.json
  collection


In [19]:
# CSV 메타데이터 있는지 확인
import subprocess
result = subprocess.run(["find", str(fsd_dir), "-name", "*.csv", "-o", "-name", "*.json"],
                        capture_output=True, text=True)
print(result.stdout)

In [20]:
# FSD50K 상위에 다른 폴더가 더 있는지
result = subprocess.run(["find", str(fsd_dir), "-maxdepth", "2", "-type", "d"],
                        capture_output=True, text=True)
print(result.stdout)

# BinauralCuratedDataset 전체 최상위도
result2 = subprocess.run(["find", str(SEMHEAR_DIR), "-maxdepth", "2", "-type", "d"],
                         capture_output=True, text=True)
print(result2.stdout)

/content/drive/MyDrive/dcase2026_task4/data/BinauralCuratedDataset/FSD50K
/content/drive/MyDrive/dcase2026_task4/data/BinauralCuratedDataset/FSD50K/FSD50K.dev_audio

/content/drive/MyDrive/dcase2026_task4/data/BinauralCuratedDataset
/content/drive/MyDrive/dcase2026_task4/data/BinauralCuratedDataset/TAU-acoustic-sounds
/content/drive/MyDrive/dcase2026_task4/data/BinauralCuratedDataset/TAU-acoustic-sounds/TAU-urban-acoustic-scenes-2019-development
/content/drive/MyDrive/dcase2026_task4/data/BinauralCuratedDataset/TAU-acoustic-sounds/TAU-urban-acoustic-scenes-2019-evaluation
/content/drive/MyDrive/dcase2026_task4/data/BinauralCuratedDataset/FSD50K
/content/drive/MyDrive/dcase2026_task4/data/BinauralCuratedDataset/FSD50K/FSD50K.dev_audio



In [27]:
# 없는 클래스 폴더를 빈 폴더로 생성
for split in ("train", "valid"):
    for cls in selected_classes:
        d = bg_dir / split / cls
        d.mkdir(parents=True, exist_ok=True)

print("빈 폴더 생성 완료")

# 확인
for split in ("train", "valid"):
    dirs = list((bg_dir / split).iterdir())
    print(f"{split}: {len(dirs)}개 폴더")

NameError: name 'selected_classes' is not defined

In [41]:
# 현재까지 처리된 파일 수 확인
import subprocess
result = subprocess.run(
    ["find", str(DEV_SET_DIR / "interference"), "-name", "*.wav"],
    capture_output=True, text=True
)
lines = result.stdout.strip().splitlines()
print(f"완료된 파일: {len(lines)}개")

완료된 파일: 1957개


In [28]:
# 46/94 class coverage 이고
# FSD50K 데이터도 wav 파일 덜 다운로드 받았지만

# 일단 new_datamodule.py이랑 new_dataset.py 작동 확인

if (DEV_SET_DIR / "interference" / "train").exists():
    print("[skip] interference/train already exists — add_interference.py skip")
else:
    result = subprocess.run([
        sys.executable, "add_interference.py",
        "--input_dir", str(bg_dir),
        "--output_dir", str(DEV_SET_DIR / "interference"),
        "--target_sr=32000", "--amp_threshold=0.02",
        "--min_length=0.1", "--segment=10", "--shift=0.1",
    ], cwd=PROJECT_ROOT, capture_output=True, text=True)

    print(result.stdout[-3000:] if result.stdout else "(no stdout)")
    if result.returncode != 0:
        print("[STDERR]", result.stderr[-2000:])
    else:
        (DEV_SET_DIR / ".interference_added").write_text("ok")
        print("add_interference.py OK")

input_dir    : /content/bg_scaper_fmt
output_dir   : /content/drive/MyDrive/dcase2026_task4/data/dev_set/interference
output_sr    : 32000
amp_threshold: 0.02
min_length   : 0.1
segment      : 10.0
shift        : 0.1
input_dir    : /content/bg_scaper_fmt/train
output_dir   : /content/drive/MyDrive/dcase2026_task4/data/dev_set/interference/train

[STDERR] Traceback (most recent call last):
  File "/content/dcase2026_task4/add_interference.py", line 147, in <module>
    prepare_data(input_dir= os.path.join(args.input_dir, 'train'),
  File "/content/dcase2026_task4/add_interference.py", line 96, in prepare_data
    assert os.path.exists(src_path) and os.path.isdir(src_path), f'There is no {src_path}'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: There is no /content/bg_scaper_fmt/train/Air conditioning



## Step 5. 경로 검증 (dataloader 스mo크용)

In [43]:
REQUIRED = [
    "data/dev_set/sound_event/train", "data/dev_set/noise/train",
    "data/dev_set/interference/train", "data/dev_set/room_ir/train",
    "data/dev_set/metadata/valid.json",
    "src/modules/spatial_audio_synthesizer/spatial_audio_synthesizer.py",
]
missing = [r for r in REQUIRED if not (PROJECT_ROOT / r).exists()]
for r in REQUIRED:
    print(f"[{'OK' if r not in missing else 'MISSING'}] {r}")
if missing:
    print("\n[warn] missing paths — Step 7–8 skip 가능, unittest는 OK")
else:
    print("\nStep 5 OK")

[OK] data/dev_set/sound_event/train
[OK] data/dev_set/noise/train
[OK] data/dev_set/interference/train
[OK] data/dev_set/room_ir/train
[OK] data/dev_set/metadata/valid.json
[OK] src/modules/spatial_audio_synthesizer/spatial_audio_synthesizer.py

Step 5 OK


---
## ★ unittest 전용 셀 (`test_new_dataset` + `test_new_datamodule`)

**dev set / Semantic Hearing / SpAudSyn 없이 실행 가능.**
Colab → Step 0 → helpers → **이 셀** 만으로도 검증 가능.

In [ ]:
def run_datamodule_tests(verbosity=2):
    loader = unittest.TestLoader()
    suite = unittest.TestSuite()
    suite.addTests(loader.discover(
        start_dir=str(PROJECT_ROOT / "tests" / "datamodules"),
        pattern="test_new_*.py",
        top_level_dir=str(PROJECT_ROOT),
    ))
    stream = StringIO()
    result = unittest.TextTestRunner(stream=stream, verbosity=verbosity).run(suite)
    print(stream.getvalue())
    print(f"\nRan {result.testsRun} | failures={len(result.failures)} | errors={len(result.errors)}")
    for label, items in [("FAILURES", result.failures), ("ERRORS", result.errors)]:
        if items:
            print(f"\n--- {label} ---")
            for test, tb in items:
                print(test, "\n", tb)
    return result

test_result = run_datamodule_tests()
assert not test_result.failures and not test_result.errors
print("\n★ unittest OK — new_dataset.py / new_datamodule.py (26 tests)")

['AlarmClock', 'BicycleBell', 'Blender', 'Buzzer', 'Clapping', 'Cough', 'CupboardOpenClose', 'Dishes', 'Doorbell', 'FootSteps', 'HairDryer', 'MechanicalFans', 'MusicalKeyboard', 'Percussion', 'Pour', 'Speech', 'Typing', 'VacuumCleaner']
18
['AlarmClock', 'BicycleBell', 'Blender', 'Buzzer', 'Clapping', 'Cough', 'CupboardOpenClose', 'Dishes', 'Doorbell', 'FootSteps', 'HairDryer', 'MechanicalFans', 'MusicalKeyboard', 'Percussion', 'Pour', 'Speech', 'Typing', 'VacuumCleaner']
18
['AlarmClock', 'BicycleBell', 'Blender', 'Buzzer', 'Clapping', 'Cough', 'CupboardOpenClose', 'Dishes', 'Doorbell', 'FootSteps', 'HairDryer', 'MechanicalFans', 'MusicalKeyboard', 'Percussion', 'Pour', 'Speech', 'Typing', 'VacuumCleaner']
18
['AlarmClock', 'BicycleBell', 'Blender', 'Buzzer', 'Clapping', 'Cough', 'CupboardOpenClose', 'Dishes', 'Doorbell', 'FootSteps', 'HairDryer', 'MechanicalFans', 'MusicalKeyboard', 'Percussion', 'Pour', 'Speech', 'Typing', 'VacuumCleaner']
18
['AlarmClock', 'BicycleBell', 'Blender',

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


['AlarmClock', 'BicycleBell', 'Blender', 'Buzzer', 'Clapping', 'Cough', 'CupboardOpenClose', 'Dishes', 'Doorbell', 'FootSteps', 'HairDryer', 'MechanicalFans', 'MusicalKeyboard', 'Percussion', 'Pour', 'Speech', 'Typing', 'VacuumCleaner']
18
['AlarmClock', 'BicycleBell', 'Blender', 'Buzzer', 'Clapping', 'Cough', 'CupboardOpenClose', 'Dishes', 'Doorbell', 'FootSteps', 'HairDryer', 'MechanicalFans', 'MusicalKeyboard', 'Percussion', 'Pour', 'Speech', 'Typing', 'VacuumCleaner']
18
['AlarmClock', 'BicycleBell', 'Blender', 'Buzzer', 'Clapping', 'Cough', 'CupboardOpenClose', 'Dishes', 'Doorbell', 'FootSteps', 'HairDryer', 'MechanicalFans', 'MusicalKeyboard', 'Percussion', 'Pour', 'Speech', 'Typing', 'VacuumCleaner']
18
['AlarmClock', 'BicycleBell', 'Blender', 'Buzzer', 'Clapping', 'Cough', 'CupboardOpenClose', 'Dishes', 'Doorbell', 'FootSteps', 'HairDryer', 'MechanicalFans', 'MusicalKeyboard', 'Percussion', 'Pour', 'Speech', 'Typing', 'VacuumCleaner']
18
['AlarmClock', 'BicycleBell', 'Blender',

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## Step 7. Train DataModule 구성 (Step 1–2·5 완료 후)

In [44]:
from src.datamodules.new_datamodule import DataModule

with open(PROJECT_ROOT / "config" / "label" / "m2dat_1c.yaml") as f:
    cfg = yaml.safe_load(f)
dm_args = cfg["datamodule"]["args"]
for split in ("train_dataloader", "val_dataloader"):
    if split in dm_args:
        ds = dm_args[split]["dataset"]
        ds["module"] = "src.datamodules.new_dataset"
        ds["main"] = "DatasetS3"

train_cfg = dm_args["train_dataloader"]
train_cfg.update(batch_size=2, num_workers=0, persistent_workers=False)
train_cfg["dataset"]["args"]["config"]["dataset_length"] = 8
val_cfg = dm_args.get("val_dataloader")
if val_cfg:
    val_cfg.update(batch_size=2, num_workers=0, persistent_workers=False)

dm = DataModule(train_dataloader=train_cfg, val_dataloader=val_cfg)
train_loader = dm.train_dataloader()
print("Step 7 OK | mode:", train_cfg["dataset"]["args"]["config"]["mode"])

['AlarmClock', 'BicycleBell', 'Blender', 'Buzzer', 'Clapping', 'Cough', 'CupboardOpenClose', 'Dishes', 'Doorbell', 'FootSteps', 'HairDryer', 'MechanicalFans', 'MusicalKeyboard', 'Percussion', 'Pour', 'Speech', 'Typing', 'VacuumCleaner']
18
['AlarmClock', 'BicycleBell', 'Blender', 'Buzzer', 'Clapping', 'Cough', 'CupboardOpenClose', 'Dishes', 'Doorbell', 'FootSteps', 'HairDryer', 'MechanicalFans', 'MusicalKeyboard', 'Percussion', 'Pour', 'Speech', 'Typing', 'VacuumCleaner']
18
Step 7 OK | mode: generate


## Step 8. Train / Val dataloader 스mo크 테스트

In [2]:
batch = next(iter(train_loader))
describe_batch(batch, "train batch")
for key in ("mixture", "labels", "doas", "active"):
    assert key in batch and batch[key].dim() >= 2
if dm.val_dataset is not None:
    describe_batch(next(iter(dm.val_dataloader())), "val batch")
print("\nStep 8 OK — dataloader smoke test passed")

NameError: name 'train_loader' is not defined